# Lightweight HBCC — CIFAR-10 Full Training Pipeline

This notebook orchestrates the full workflow implemented in this repository:

1. Environment and smoke checks
2. Teacher/reference training
3. CoC baseline training
4. Student CE-only training
5. Short proxy architecture search
6. Technique ablations
7. Knowledge distillation
8. Structured pruning export and fine-tune
9. Benchmark matrix and Pareto summary
10. Published results table

The default flags are conservative. Turn on the phases you want to run before executing all cells.

CIFAR-10 training uses `train=45k` and `val=5k` from the original training split. The official CIFAR-10 test split is evaluated once on `best.pth` and reported as `test_acc1`. See `hbcc-cifar-100.ipynb` for the matching CIFAR-100 Kaggle pipeline.


In [1]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster


Cloning into 'HBCC'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (351/351), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 351 (delta 232), reused 340 (delta 228), pack-reused 0 (from 0)
Receiving objects: 100% (351/351), 8.17 MiB | 27.43 MiB/s, done.
Resolving deltas: 100% (232/232), done.
/kaggle/working/HBCC


In [2]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Run this notebook from the repo root or notebooks dir. Got {ROOT}"

COC_PYTHON = Path(r"D:\Anaconda\envs\CoC\python.exe")
PYTHON = COC_PYTHON if COC_PYTHON.exists() else Path(sys.executable)

print("Repo:", ROOT)
print("Python:", PYTHON)

Repo: /kaggle/working/HBCC
Python: /usr/bin/python3


## Phase Switches

Set the flags below. For a first run, keep only `RUN_SMOKE = True`. For the full research run, enable each phase deliberately.

In [3]:
RUN_ENV_CHECKS = True
RUN_SMOKE = False

# Session 2: train teacher + KD models only; baselines were trained in session 1.
RUN_TEACHER = True
RUN_LIGHTWEIGHT_BASELINES = False  # already trained; results loaded from CSV
RUN_COC_BASELINE = False           # already trained; results loaded from CSV
RUN_STUDENT_CE = False  # HBCC already trained; skip to save ~5h
RUN_PROXY_SEARCH = False
RUN_ABLATIONS = False
RUN_KD = True
RUN_PRUNING = False  # requires HBCC checkpoint from same session
RUN_BENCHMARKS = True
RUN_PARETO_REPORT = True

# Epoch budgets — carefully sized to stay within Kaggle 12h GPU limit:
# Teacher(300ep,2.67h) + KD-Medium(200ep,1.61h) + KD-Small(150ep,1.21h) + Benchmarks(0.78h) ≈ 6.27h
FULL_EPOCHS = 300
ACCURACY_EPOCHS = 300
KD_MEDIUM_EPOCHS = 200  # ~1.61h on T4
KD_SMALL_EPOCHS = 150   # ~1.21h on T4
PROXY_EPOCHS = 30
PRUNING_EPOCHS = 80
PRUNED_FINETUNE_EPOCHS = 80

COMMON_TRAIN_OVERRIDES = [
    # Reduce these if you hit OOM.
    # "data.batch_size=64",
    # "data.val_batch_size=128",
    # "data.test_batch_size=128",
]

BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP = 30
BENCHMARK_RUNS = 100

# Use shorter benchmark settings while debugging the notebook.
DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP = 2
DEBUG_BENCHMARK_RUNS = 3


In [4]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    """Run a repo Python command; tqdm updates are kept to one notebook line."""
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def train(config: str, output: str, overrides: list[str] | None = None, extra: list[str] | None = None) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args([*(overrides or []), *COMMON_TRAIN_OVERRIDES])
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
    experiment_name: str | None = None,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup = DEBUG_BENCHMARK_WARMUP if debug else BENCHMARK_WARMUP
    runs = DEBUG_BENCHMARK_RUNS if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py",
        "--config",
        config,
        "--output",
        "results/benchmark",
        "--batch-sizes",
        *[str(v) for v in batches],
        "--warmup",
        str(warmup),
        "--runs",
        str(runs),
    ]
    if experiment_name:
        args += override_args([f"experiment.name={experiment_name}"])
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)


## 0. Environment and Smoke Checks

In [5]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_latency_tiny.yaml", "--batch-size", "1"])
    run_py(["tools/shape_trace.py", "--config", "configs/hbcc_current_reference.yaml", "--batch-size", "1"])

if RUN_SMOKE:
    train(
        "configs/smoke.yaml",
        "runs_smoke",
        extra=["--limit-train-batches", "1", "--limit-val-batches", "1", "--limit-test-batches", "1"],
    )
    benchmark("configs/smoke.yaml", debug=True, profile=False)


$ /usr/bin/python3 -m pytest


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working/HBCC
configfile: pyproject.toml
testpaths: tests
plugins: langsmith-0.7.30, anyio-4.13.0, typeguard-4.5.1
collected 10 items

tests/test_config_and_pruning.py ..                                      [ 20%]
tests/test_data.py ....                                                  [ 60%]
tests/test_models.py ....                                                [100%]

============================= 10 passed in 12.60s ==============================

[exit=0] elapsed=16.9s

$ /usr/bin/python3 tools/shape_trace.py --config configs/hbcc_latency_tiny.yaml --batch-size 1


stem                                             -> (1, 32, 16, 16)
stages.0                                         -> (1, 32, 16, 16)
downsamples.0                                    -> (1, 48, 8, 8)
stages.1.blocks.0.cluster                        -> (1, 24, 8, 8)
stages.1                                         -> (1, 48, 8, 8)
downsamples.1                                    -> (1, 96, 4, 4)
stages.2.blocks.0.cluster                        -> (1, 96, 4, 4)
stages.2.blocks.1.cluster                        -> (1, 96, 4, 4)
stages.2                                         -> (1, 96, 4, 4)
downsamples.2                                    -> (1, 160, 2, 2)
stages.3.blocks.0.cluster                        -> (1, 160, 2, 2)
stages.3                                         -> (1, 160, 2, 2)
logits                                           -> (1, 10)

[exit=0] elapsed=5.5s

$ /usr/bin/python3 tools/shape_trace.py --config configs/hbcc_current_reference.yaml --batch-size 1


stem                                             -> (1, 32, 32, 32)
stages.0.blocks.0.cluster                        -> (1, 32, 32, 32)
stages.0                                         -> (1, 32, 32, 32)
downsamples.0                                    -> (1, 64, 16, 16)
stages.1.blocks.0.cluster                        -> (1, 32, 16, 16)
stages.1                                         -> (1, 64, 16, 16)
downsamples.1                                    -> (1, 128, 8, 8)
stages.2.blocks.0.cluster                        -> (1, 128, 8, 8)
stages.2.blocks.1.cluster                        -> (1, 128, 8, 8)
stages.2                                         -> (1, 128, 8, 8)
downsamples.2                                    -> (1, 192, 4, 4)
stages.3.blocks.0.cluster                        -> (1, 192, 4, 4)
stages.3                                         -> (1, 192, 4, 4)
logits                                           -> (1, 10)

[exit=0] elapsed=4.6s


## 1. Teacher and Reference Baselines

Train ResNet-18 first. It is both the accuracy reference and the default KD teacher.

In [6]:
if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        "runs_teacher",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        "runs_baselines",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        "runs_baselines",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )


$ /usr/bin/python3 tools/train.py --config configs/baselines/resnet18_cifar.yaml --output runs_teacher --override train.epochs=200


```text
test: 100%|████████████| 40/40 [00:01<00:00, 25.54it/s]
```

setup: dataset=cifar10 output=runs_teacher/resnet18_cifar device=cuda
setup: building data loaders...

setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2684.9s
setup: building model...
setup: model ready params=11173962 elapsed=2685.3s
train: starting epochs=200 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.5880141764284879,
  "train_acc1": 49.47026353276353,
  "train_time_s": 17.012847188999785,
  "val_loss": 1.2735681091308593,
  "val_acc1": 54.8,
  "val_time_s": 0.8529305150000255,
  "val_acc5": 92.65999985351563
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.214951328232757,
  "train

```text
test:  72%|████████▋   | 29/40 [00:01<00:00, 27.49it/s]
```

setup: dataset=cifar10 output=runs_baselines/mobilenet_v2_cifar device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.5s
setup: building model...
setup: model ready params=2236682 elapsed=2.8s
train: starting epochs=200 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.9293007864232077,
  "train_acc1": 31.79309116809117,
  "train_time_s": 18.93333476099997,
  "val_loss": 1.5665754159927368,
  "val_acc1": 43.76000001831055,
  "val_time_s": 3.2563327580000987,
  "val_acc5": 89.87999989013672
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.652872238403711

```text
test:  70%|████████▍   | 28/40 [00:01<00:00, 28.35it/s]
```

setup: dataset=cifar10 output=runs_baselines/shufflenet_v2_x1_0_cifar device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.4s
setup: building model...
setup: model ready params=1263854 elapsed=2.7s
train: starting epochs=200 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.8567136074403072,
  "train_acc1": 35.583600427350426,
  "train_time_s": 17.72166913000001,
  "val_loss": 1.4842819692611695,
  "val_acc1": 47.079999993896486,
  "val_time_s": 2.2654683029995795,
  "val_acc5": 91.35999992675781
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.5780022

## 2. CoC Baseline

Train the CIFAR-adapted CoC baseline separately so benchmark records can use its trained checkpoint instead of an untrained model.

In [7]:
COC_BASELINE_RUN = "runs_coc_baseline/coc_cifar_baseline"
COC_BASELINE_CHECKPOINT = f"{COC_BASELINE_RUN}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        "runs_coc_baseline",
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    run_py(["tools/summarize_training.py", COC_BASELINE_RUN])



$ /usr/bin/python3 tools/train.py --config configs/coc_cifar_baseline.yaml --output runs_coc_baseline --override train.epochs=200


```text
test:  92%|███████████ | 37/40 [00:01<00:00, 25.65it/s]
```

setup: dataset=cifar10 output=runs_coc_baseline/coc_cifar_baseline device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.4s
setup: building model...
setup: model ready params=2395814 elapsed=2.7s
train: starting epochs=200 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0009999383162408303,
  "train_loss": 1.7780395413396026,
  "train_acc1": 39.96171652421653,
  "train_time_s": 19.95590687799995,
  "val_loss": 1.359705919456482,
  "val_acc1": 51.83999995727539,
  "val_time_s": 1.0710411250001926,
  "val_acc5": 93.0999999633789
}

                                                                                     

                                                     
{
  "epoch": 1,
  "lr": 0.0009997532801828658,
  "train_loss": 1.531172518376951

Best val: epoch=199, val_acc1=90.04, val_loss=0.4319 val_acc5=97.84
Held-out test: checkpoint=best.pth, epoch=199, test_acc1=88.34, test_loss=0.4706 test_acc5=97.64

[exit=0] elapsed=0.1s


## 3. Student CE-only Training

This is the fair baseline for each HBCC student before applying KD or pruning.

In [8]:
if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_accuracy",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_accuracy",
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )


$ /usr/bin/python3 tools/train.py --config configs/hbcc_accuracy_small.yaml --output runs_accuracy --override train.epochs=300


```text
test:  90%|██████████▊ | 36/40 [00:01<00:00, 23.90it/s]
```

setup: dataset=cifar10 output=runs_accuracy/hbcc_accuracy_small device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.5s
setup: building model...
setup: model ready params=1538618 elapsed=2.7s
train: starting epochs=300 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.000208,
  "train_loss": 2.248621419624046,
  "train_acc1": 15.280003561253562,
  "train_time_s": 28.680932205999852,
  "val_loss": 2.0094475605010986,
  "val_acc1": 27.63999997253418,
  "val_time_s": 1.6530138199996145,
  "val_acc5": 79.59999992675782
}

                                                                                     

                                                      
{
  "epoch": 1,
  "lr": 0.00040599999999999995,
  "train_loss": 2.0503450126050207,
  "train

```text
test:  75%|█████████   | 30/40 [00:01<00:00, 20.20it/s]
```

setup: dataset=cifar10 output=runs_accuracy/hbcc_accuracy_medium device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.4s
setup: building model...
setup: model ready params=2840862 elapsed=2.7s
train: starting epochs=300 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.000208,
  "train_loss": 2.2325073517965115,
  "train_acc1": 15.607193732193732,
  "train_time_s": 30.175952237001184,
  "val_loss": 1.975362889099121,
  "val_acc1": 27.97999999084473,
  "val_time_s": 1.8328413109993562,
  "val_acc5": 80.80000004882812
}

                                                                                     

                                                     
{
  "epoch": 1,
  "lr": 0.00040599999999999995,
  "train_loss": 2.0392502892730584,
  "train

## 4. Short Proxy Architecture Search

Use 20-50 epochs to reject weak configurations. Only top candidates should get full training.

In [9]:
PROXY_JOBS = [
    {
        "name": "proxy_dims24_depth1111_mlp2",
        "overrides": [
            "model.embed_dims=[24,48,96,160]",
            "model.depths=[1,1,1,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_depth1121_mlp2",
        "overrides": [
            "model.embed_dims=[32,48,96,160]",
            "model.depths=[1,1,2,1]",
            "model.mlp_ratios=2.0",
        ],
    },
    {
        "name": "proxy_dims32_64_depth1221_mlp3",
        "overrides": [
            "model.embed_dims=[32,64,128,192]",
            "model.depths=[1,2,2,1]",
            "model.mlp_ratios=3.0",
            "model.heads=[1,2,4,4]",
        ],
    },
]

if RUN_PROXY_SEARCH:
    for job in PROXY_JOBS:
        train(
            "configs/hbcc_latency_tiny.yaml",
            "runs_proxy",
            overrides=[
                f"experiment.name={job['name']}",
                f"train.epochs={PROXY_EPOCHS}",
                *job["overrides"],
            ],
        )

## 5. Independent Ablations

Run each technique independently and decide `keep`, `drop`, or `defer` from metrics.

In [10]:
ABLATION_CONFIGS = [
    "configs/ablations/hbcc_tiny_hamming_late.yaml",
    "configs/ablations/hbcc_tiny_lbpconv.yaml",
]

if RUN_ABLATIONS:
    for cfg in ABLATION_CONFIGS:
        train(cfg, "runs_ablations", overrides=[f"train.epochs={FULL_EPOCHS}"])

## 6. Knowledge Distillation

Requires a trained teacher checkpoint. The teacher is used only during training; inference latency of the student is unchanged.

In [11]:
TEACHER_CHECKPOINT = "runs_teacher/resnet18_cifar/best.pth"

# Timing estimate: Teacher(2.67h) + KD-Tiny(~0.9h) + KD-Medium(1.61h) + KD-Small(1.21h) + Benchmarks(0.78h) ≈ 7.17h
if RUN_KD:
    if not (ROOT / TEACHER_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train the teacher first: {TEACHER_CHECKPOINT}")
    KD_EXTRA = [
        "--teacher-config", "configs/baselines/resnet18_cifar.yaml",
        "--teacher-checkpoint", TEACHER_CHECKPOINT,
    ]
    KD_BASE = ["train.kd_alpha=0.5", "train.kd_temperature=4.0"]

    # Latency-Tiny KD (~0.9h at 300 epochs)
    train(
        "configs/hbcc_latency_tiny.yaml",
        "runs_kd",
        overrides=[f"train.epochs={FULL_EPOCHS}", *KD_BASE],
        extra=KD_EXTRA,
    )

    # Accuracy-Medium KD (~1.61h at 200 epochs) — primary CIFAR-10/100 uplift target
    train(
        "configs/hbcc_accuracy_medium.yaml",
        "runs_kd",
        overrides=[
            f"train.epochs={KD_MEDIUM_EPOCHS}",
            "experiment.name=hbcc_accuracy_medium_kd",
            *KD_BASE,
        ],
        extra=KD_EXTRA,
    )

    # Accuracy-Small KD (~1.21h at 150 epochs)
    train(
        "configs/hbcc_accuracy_small.yaml",
        "runs_kd",
        overrides=[
            f"train.epochs={KD_SMALL_EPOCHS}",
            "experiment.name=hbcc_accuracy_small_kd",
            *KD_BASE,
        ],
        extra=KD_EXTRA,
    )


## 7. Structured Pruning Export

Do not judge latency from the soft-mask model. Export a materialized smaller config, fine-tune it, then benchmark the exported model.

In [12]:
PRUNING_BASE_CHECKPOINT = "runs_accuracy/hbcc_accuracy_small/best.pth"
PRUNING_CONFIG = "configs/ablations/hbcc_accuracy_small_pruning_mask.yaml"
PRUNING_RUN_DIR = "runs_pruning_accuracy"
PRUNING_EXPERIMENT = "hbcc_accuracy_small_pruning_mask"
PRUNING_CHECKPOINT = f"{PRUNING_RUN_DIR}/{PRUNING_EXPERIMENT}/best.pth"
PRUNING_THRESHOLD = "0.90"
PRUNED_CONFIG = "configs/generated_ablations/hbcc_accuracy_small_pruned_export.yaml"
PRUNED_FINETUNE_RUN_DIR = "runs_pruned_accuracy"
PRUNED_FINETUNE_CHECKPOINT = "runs_pruned_accuracy/hbcc_accuracy_small_pruned_export/best.pth"

if RUN_PRUNING:
    if not (ROOT / PRUNING_BASE_CHECKPOINT).exists():
        raise FileNotFoundError(f"Train hbcc_accuracy_small first: {PRUNING_BASE_CHECKPOINT}")
    train(
        PRUNING_CONFIG,
        PRUNING_RUN_DIR,
        overrides=[f"train.epochs={PRUNING_EPOCHS}"],
        extra=["--resume", PRUNING_BASE_CHECKPOINT],
    )
    run_py([
        "tools/inspect_pruning_masks.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--thresholds",
        "0.5",
        "0.7",
        "0.8",
        "0.9",
        "0.95",
    ])
    run_py([
        "tools/export_pruned.py",
        "--config",
        PRUNING_CONFIG,
        "--checkpoint",
        PRUNING_CHECKPOINT,
        "--output",
        PRUNED_CONFIG,
        "--threshold",
        PRUNING_THRESHOLD,
        "--min-channels",
        "16",
        "--round-to",
        "8",
    ])
    train(
        PRUNED_CONFIG,
        PRUNED_FINETUNE_RUN_DIR,
        overrides=[f"train.epochs={PRUNED_FINETUNE_EPOCHS}"],
    )



$ /usr/bin/python3 tools/train.py --config configs/ablations/hbcc_accuracy_small_pruning_mask.yaml --output runs_pruning_accuracy --override train.epochs=80 --resume runs_accuracy/hbcc_accuracy_small/best.pth


```text
test:  88%|██████████▌ | 35/40 [00:01<00:00, 22.49it/s]
```

setup: dataset=cifar10 output=runs_pruning_accuracy/hbcc_accuracy_small_pruning_mask device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.7s
setup: building model...
setup: model ready params=1539130 elapsed=2.9s
setup: loading checkpoint runs_accuracy/hbcc_accuracy_small/best.pth
train: starting epochs=80 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0001999229036240723,
  "train_loss": 0.6281819599985736,
  "train_acc1": 94.89850427350427,
  "train_time_s": 33.73790300199471,
  "val_loss": 0.27336869492530824,
  "val_acc1": 93.37999985351563,
  "val_time_s": 1.783451855997555,
  "val_acc5": 99.67999989013671
}

                                                                                     

                                              

stage	channels	min	max	mean	keep@0.5	rounded_keep@0.5	keep@0.7	rounded_keep@0.7	keep@0.8	rounded_keep@0.8	keep@0.9	rounded_keep@0.9	keep@0.95	rounded_keep@0.95
0	48	0.9689	0.9705	0.9699	48	48	48	48	48	48	48	48	48	48
1	80	0.9688	0.9709	0.9700	80	80	80	80	80	80	80	80	80	80
2	160	0.9688	0.9709	0.9700	160	160	160	160	160	160	160	160	160	160
3	224	0.9283	0.9723	0.9554	224	224	224	224	224	224	224	224	149	152

[exit=0] elapsed=4.5s

$ /usr/bin/python3 tools/export_pruned.py --config configs/ablations/hbcc_accuracy_small_pruning_mask.yaml --checkpoint runs_pruning_accuracy/hbcc_accuracy_small_pruning_mask/best.pth --output configs/generated_ablations/hbcc_accuracy_small_pruned_export.yaml --threshold 0.90 --min-channels 16 --round-to 8


{'experiment': {'name': 'hbcc_accuracy_small_pruned_export'}, 'data': {'name': 'cifar10', 'root': 'data', 'download': True, 'batch_size': 128, 'val_batch_size': 256, 'test_batch_size': 256, 'val_size': 5000, 'split_seed': 42, 'workers': 2, 'augment': True, 'randaugment': {'enabled': True, 'num_ops': 2, 'magnitude': 9}, 'random_erasing': {'p': 0.25, 'scale': [0.02, 0.2], 'ratio': [0.3, 3.3]}}, 'model': {'name': 'hbcc', 'num_classes': 10, 'use_coord': True, 'embed_dims': [48, 80, 160, 224], 'depths': [2, 2, 3, 1], 'mlp_ratios': 3.0, 'heads': [2, 2, 4, 4], 'head_dim': [16, 16, 16, 16], 'proposals': [[2, 2], [2, 2], [2, 2], [2, 2]], 'folds': [[4, 4], [2, 2], [1, 1], [1, 1]], 'similarities': ['cosine', 'cosine', 'cosine', 'cosine'], 'stage_modes': ['hybrid', 'hybrid', 'cluster', 'cluster'], 'local_branches': ['lbpconv', 'dwconv', 'identity', 'identity'], 'local_ratios': [0.5, 0.5, 0.0, 0.0], 'channel_shuffle': [True, True, False, False], 'norm': 'bn', 'stem_stride': 2, 'drop_path_rate': 0.0

```text
test:  85%|██████████▏ | 34/40 [00:01<00:00, 21.85it/s]
```

setup: dataset=cifar10 output=runs_pruned_accuracy/hbcc_accuracy_small_pruned_export device=cuda
setup: building data loaders...
setup: data ready train_samples=45000 train_batches=351 val_samples=5000 val_batches=20 test_enabled=True elapsed=2.7s
setup: building model...
setup: model ready params=1538618 elapsed=2.9s
train: starting epochs=80 skip_test=False

                                                                                     

                                                      
{
  "epoch": 0,
  "lr": 0.0001999229036240723,
  "train_loss": 1.9209053190345438,
  "train_acc1": 33.90758547008547,
  "train_time_s": 31.43347849100246,
  "val_loss": 1.4373950004577636,
  "val_acc1": 50.439999938964846,
  "val_time_s": 1.7845011219978915,
  "val_acc5": 91.18
}

                                                                                     

                                                     
{
  "epoch": 1,
  "lr": 0.0001996917333733128,
  "train_loss": 1.7263662

## 8. Benchmark Matrix

Run this after training checkpoints exist. Missing checkpoints are skipped and the benchmark falls back to untrained weights only when no checkpoint is provided.

In [13]:
BENCHMARK_JOBS = [
    # (config, experiment_name_override, checkpoint)  — use None to keep config default name
    ("configs/baselines/resnet18_cifar.yaml",            None, "runs_teacher/resnet18_cifar/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml",         None, "runs_baselines/mobilenet_v2_cifar/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml",  None, "runs_baselines/shufflenet_v2_x1_0_cifar/best.pth"),
    ("configs/coc_cifar_baseline.yaml",                   None, COC_BASELINE_CHECKPOINT),
    ("configs/hbcc_current_reference.yaml",               None, None),
    ("configs/hbcc_latency_tiny.yaml",                    None, "runs_students/hbcc_latency_tiny/best.pth"),
    ("configs/hbcc_latency_small.yaml",                   None, "runs_students/hbcc_latency_small/best.pth"),
    ("configs/hbcc_accuracy_small.yaml",                  None, "runs_accuracy/hbcc_accuracy_small/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",                 None, "runs_accuracy/hbcc_accuracy_medium/best.pth"),
    # KD variants: experiment_name override prevents overwriting base model benchmark JSONs
    ("configs/hbcc_latency_tiny.yaml",   "hbcc_latency_tiny_kd",    "runs_kd/hbcc_latency_tiny/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml","hbcc_accuracy_medium_kd", "runs_kd/hbcc_accuracy_medium_kd/best.pth"),
    ("configs/hbcc_accuracy_small.yaml", "hbcc_accuracy_small_kd",  "runs_kd/hbcc_accuracy_small_kd/best.pth"),
    ("configs/ablations/hbcc_tiny_hamming_late.yaml", None, "runs_ablations/hbcc_tiny_hamming_late/best.pth"),
    ("configs/ablations/hbcc_tiny_lbpconv.yaml",      None, "runs_ablations/hbcc_tiny_lbpconv/best.pth"),
    (PRUNED_CONFIG, None, PRUNED_FINETUNE_CHECKPOINT),
]

if RUN_BENCHMARKS:
    for cfg, exp_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print("Skip missing config:", cfg)
            continue
        benchmark(cfg, checkpoint=ckpt, experiment_name=exp_name, debug=False, profile=True)



$ /usr/bin/python3 tools/benchmark.py --config configs/baselines/resnet18_cifar.yaml --output results/benchmark --batch-sizes 1 16 64 128 --warmup 30 --runs 100 --checkpoint runs_teacher/resnet18_cifar/best.pth --profile


/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "resnet18_cifar",
  "config_id": "resnet18_cifar",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 11173962,
  "params_trainable": 11173962,
  "model_size_mb": 42.66205596923828,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 3.059295880011632,
  "throughput_b1": 481.7724315290857,
  "latency_ms_b16": 5.767535359991598,
  "throughput_b16": 2841.279198784183,
  "latency_ms_b64": 15.763442579991533,
  "throughput_b64": 4054.5958297288744,
  "latency_ms_b128": 30.243530250008916,
  "throughput_b128": 4166.6

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "mobilenet_v2_cifar",
  "config_id": "mobilenet_v2_cifar",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 2236682,
  "params_trainable": 2236682,
  "model_size_mb": 8.662788391113281,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 5.3330457500123885,
  "throughput_b1": 187.06884652906695,
  "latency_ms_b16": 5.424723420001101,
  "throughput_b16": 3136.6687015993575,
  "latency_ms_b64": 6.9430141700286185,
  "throughput_b64": 9288.10330818941,
  "latency_ms_b128": 13.850148409983376,
  "throughput_b128":

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "shufflenet_v2_x1_0_cifar",
  "config_id": "shufflenet_v2_x1_0_cifar",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 1263854,
  "params_trainable": 1263854,
  "model_size_mb": 4.883369445800781,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 6.5253828200366115,
  "throughput_b1": 156.62171447508865,
  "latency_ms_b16": 7.146653090021573,
  "throughput_b16": 2317.846824712107,
  "latency_ms_b64": 7.079031339962967,
  "throughput_b64": 9155.679397142656,
  "latency_ms_b128": 14.418299440003466,
  "throug

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "coc_cifar_baseline",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 2395814,
  "params_trainable": 2395814,
  "model_size_mb": 9.139305114746094,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 12.46992941996723,
  "throughput_b1": 79.84775524331896,
  "latency_ms_b16": 12.902566669945372,
  "throughput_b16": 1240.587516161726,
  "latency_ms_b64": 12.821772960014641,
  "throughput_b64": 4950.21800117096,
  "latency_ms_b128": 14.888264119945234,
  "throughput_b128": 8676.5064133041

...

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_accuracy_medium",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 2840862,
  "params_trainable": 2836254,
  "model_size_mb": 10.8741455078125,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 15.478922889960813,
  "throughput_b1": 64.14815510118568,
  "latency_ms_b16": 15.974119230013457,
  "throughput_b16": 1016.579008368738,
  "latency_ms_b64": 16.37810025997169,
  "throughput_b64": 3909.192120409594,
  "latency_ms_b128": 22.18380073994922,
  "throughput_b128": 5781.491545180

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_latency_tiny",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 447746,
  "params_trainable": 447746,
  "model_size_mb": 1.719390869140625,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 7.309727370011387,
  "throughput_b1": 131.66417549439126,
  "latency_ms_b16": 8.054120960005093,
  "throughput_b16": 1977.3549822279037,
  "latency_ms_b64": 7.324346739987959,
  "throughput_b64": 8765.780442218493,
  "latency_ms_b128": 7.472662650034181,
  "throughput_b128": 17533.56103697945,

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_tiny_hamming_late",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 447746,
  "params_trainable": 447746,
  "model_size_mb": 1.719390869140625,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 5120,
  "latency_ms_b1": 7.630177199971513,
  "throughput_b1": 133.7382950310159,
  "latency_ms_b16": 7.57064410005114,
  "throughput_b16": 2103.709070494404,
  "latency_ms_b64": 7.630881549994228,
  "throughput_b64": 7933.101326609576,
  "latency_ms_b128": 7.880719429958845,
  "throughput_b128": 16366.1459331

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_tiny_lbpconv",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 463258,
  "params_trainable": 459226,
  "model_size_mb": 1.78155517578125,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 7.570573029952357,
  "throughput_b1": 142.07456772300444,
  "latency_ms_b16": 7.066794409984141,
  "throughput_b16": 2248.037596305106,
  "latency_ms_b64": 7.208657730006962,
  "throughput_b64": 8959.143731596456,
  "latency_ms_b128": 7.150873560021864,
  "throughput_b128": 17770.33780630816,
 

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
{
  "model_name": "hbcc",
  "config_id": "hbcc_accuracy_small_pruned_export",
  "device": "cuda",
  "device_name": "Tesla T4",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8",
  "image_size": 32,
  "params_total": 1538618,
  "params_trainable": 1535162,
  "model_size_mb": 5.8943634033203125,
  "macs": null,
  "flops": null,
  "unsupported_ops": null,
  "flop_error": "ModuleNotFoundError(\"No module named 'fvcore'\")",
  "bops": 0,
  "latency_ms_b1": 13.007071430038195,
  "throughput_b1": 77.17651652956738,
  "latency_ms_b16": 13.241482520024874,
  "throughput_b16": 1212.6022872792641,
  "latency_ms_b64": 13.424425150005845,
  "throughput_b64": 4832.46032218282,
  "latency_ms_b128": 16.752616129961098,
  "throughput_b128

## 9. Read Metrics and Build Summary Tables

In [14]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for metrics_path in ROOT.glob("runs*/**/metrics.jsonl"):
        records = read_jsonl(metrics_path)
        val_records = [r for r in records if "val_acc1" in r]
        if not val_records:
            continue
        best = max(val_records, key=lambda r: r["val_acc1"])
        test_records = [r for r in records if "test_acc1" in r]
        test_record = next((r for r in reversed(test_records) if r.get("epoch") == best.get("epoch")), test_records[-1] if test_records else {})
        train_records = [r for r in records if "train_acc1" in r]
        last_train = train_records[-1] if train_records else {}
        rows.append({
            "run_dir": str(metrics_path.parent.relative_to(ROOT)),
            "experiment": metrics_path.parent.name,
            "best_epoch": best.get("epoch"),
            "best_val_acc1": best.get("val_acc1"),
            "best_val_loss": best.get("val_loss"),
            "test_acc1": test_record.get("test_acc1"),
            "test_loss": test_record.get("test_loss"),
            "last_epoch": last_train.get("epoch"),
            "last_train_acc1": last_train.get("train_acc1"),
        })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


train_df = collect_train_metrics()
train_df

,run_dir,experiment,best_epoch,best_val_acc1,best_val_loss,test_acc1,test_loss,last_epoch,last_train_acc1
7,runs_teacher/resnet18_cifar,resnet18_cifar,191,94.52,0.283018,94.16,0.298616,199,99.997774
3,runs_accuracy/hbcc_accuracy_medium,hbcc_accuracy_medium,272,94.50,0.394306,94.45,0.399384,299,60.906339
5,runs_pruning_accuracy/hbcc_accuracy_small_prun...,hbcc_accuracy_small_pruning_mask,68,94.36,0.257532,93.88,0.263033,79,96.808226
4,runs_accuracy/hbcc_accuracy_small,hbcc_accuracy_small,290,94.22,0.417829,94.00,0.427991,299,58.411236
2,runs_baselines/shufflenet_v2_x1_0_cifar,shufflenet_v2_x1_0_cifar,170,92.46,0.341933,91.68,0.369031,199,99.957710
1,runs_baselines/mobilenet_v2_cifar,mobilenet_v2_cifar,198,91.98,0.376906,91.04,0.402075,199,99.857550
0,runs_coc_baseline/coc_cifar_baseline,coc_cifar_baseline,199,90.04,0.431868,88.34,0.470636,199,99.819712
6,runs_pruned_accuracy/hbcc_accuracy_small_prune...,hbcc_accuracy_small_pruned_export,77,87.36,0.437043,86.48,0.450695,79,82.988337


In [15]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / "results" / "benchmark").glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rec["benchmark_file"] = str(path.relative_to(ROOT))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
cols = [
    "config_id",
    "model_name",
    "params_total",
    "macs",
    "bops",
    "latency_ms_b1",
    "throughput_b16",
    "throughput_b64",
    "throughput_b128",
    "peak_memory_mb",
]
bench_df[[c for c in cols if c in bench_df.columns]] if not bench_df.empty else bench_df

,config_id,model_name,params_total,macs,bops,latency_ms_b1,throughput_b16,throughput_b64,throughput_b128,peak_memory_mb
0,hbcc_tiny_hamming_late,hbcc,447746,None,5120,7.630177,2103.709070,7933.101327,16366.145933,44.390625
1,hbcc_accuracy_small_pruned_export,hbcc,1538618,None,0,13.007071,1212.602287,4832.460322,7680.908223,109.592285
2,hbcc_tiny_lbpconv,hbcc,463258,None,0,7.570573,2248.037596,8959.143732,17770.337806,116.450195
3,hbcc_accuracy_medium,hbcc,2840862,None,0,15.478923,1016.579008,3909.192120,5781.491545,145.565918
4,coc_cifar_baseline,hbcc,2395814,None,0,12.469929,1240.587516,4950.218001,8676.506413,71.798828
5,mobilenet_v2_cifar,mobilenet_v2_cifar,2236682,None,0,5.333046,3136.668702,9288.103308,9256.085863,123.365234
6,hbcc_current_reference,hbcc,870100,None,0,7.676887,2029.235433,7003.164030,7222.403412,173.993164
7,hbcc_latency_tiny,hbcc,447746,None,0,7.309727,1977.354982,8765.780442,17533.561037,44.390625
8,hbcc_latency_small,hbcc,902932,None,0,8.986031,1797.444073,6912.642748,13902.589064,54.133301
9,hbcc_accuracy_small,hbcc,1538618,None,0,13.745859,1173.894303,4655.412038,7506.428304,109.592285


## 10. Pareto Summary

This cell combines training accuracy with benchmark records when names match. Review the mapping before using it as a final report.

In [16]:
def attach_accuracy(bench: pd.DataFrame, train_metrics: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return bench
    out = bench.copy()
    out["acc1"] = None
    # Fall back to committed published results for models not trained this session.
    # This lets baselines-only and HBCC-only runs each produce a complete table.
    fallback_csv = ROOT / "results" / "cifar10_published_results.csv"
    fallback_map: dict = {}
    if fallback_csv.exists():
        try:
            fb = pd.read_csv(fallback_csv)
            if "Model" in fb.columns and "Top-1 Acc (%)" in fb.columns:
                fallback_map = dict(zip(fb["Model"], fb["Top-1 Acc (%)"]))
        except Exception:
            pass
    if not train_metrics.empty:
        acc_source = "test_acc1" if "test_acc1" in train_metrics.columns else "best_val_acc1"
        acc_map = dict(zip(train_metrics["experiment"], train_metrics[acc_source].fillna(train_metrics["best_val_acc1"])))
    else:
        acc_map = {}
    for idx, row in out.iterrows():
        name = row.get("config_id")
        out.at[idx, "acc1"] = acc_map.get(name, fallback_map.get(name))
    return out


def pareto_frontier(df: pd.DataFrame) -> pd.DataFrame:
    required = ["acc1", "params_total", "latency_ms_b1", "peak_memory_mb"]
    if df.empty or any(c not in df.columns for c in required):
        return pd.DataFrame()
    valid = df.dropna(subset=required).copy()
    keep = []
    for i, row in valid.iterrows():
        dominated = False
        for j, other in valid.iterrows():
            if i == j:
                continue
            better_or_equal = (
                other["acc1"] >= row["acc1"]
                and other["params_total"] <= row["params_total"]
                and other["latency_ms_b1"] <= row["latency_ms_b1"]
                and other["peak_memory_mb"] <= row["peak_memory_mb"]
            )
            strictly_better = (
                other["acc1"] > row["acc1"]
                or other["params_total"] < row["params_total"]
                or other["latency_ms_b1"] < row["latency_ms_b1"]
                or other["peak_memory_mb"] < row["peak_memory_mb"]
            )
            if better_or_equal and strictly_better:
                dominated = True
                break
        if not dominated:
            keep.append(i)
    return valid.loc[keep].sort_values(["acc1", "latency_ms_b1"], ascending=[False, True])


combined_df = attach_accuracy(bench_df, train_df)
frontier_df = pareto_frontier(combined_df)

out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
if not combined_df.empty:
    combined_df.to_csv(out_dir / "combined_training_benchmark.csv", index=False)
if not frontier_df.empty:
    frontier_df.to_csv(out_dir / "pareto_frontier.csv", index=False)

frontier_df[[c for c in ["config_id", "acc1", "params_total", "macs", "bops", "latency_ms_b1", "peak_memory_mb"] if c in frontier_df.columns]]

,config_id,acc1,params_total,macs,bops,latency_ms_b1,peak_memory_mb
3,hbcc_accuracy_medium,94.45,2840862,None,0,15.478923,145.565918
11,resnet18_cifar,94.16,11173962,None,0,3.059296,326.364746
9,hbcc_accuracy_small,94.0,1538618,None,0,13.745859,109.592285
10,shufflenet_v2_x1_0_cifar,91.68,1263854,None,0,6.525383,94.832520
5,mobilenet_v2_cifar,91.04,2236682,None,0,5.333046,123.365234
4,coc_cifar_baseline,88.34,2395814,None,0,12.469929,71.798828


## 11. Published Results

Final headline table for reporting: Top-1 accuracy alongside params, MACs, BOPs, batch-1 latency, and peak memory for every model that has both a training run and a benchmark record. Sorted by accuracy, descending. This is the table to cite in the paper.


In [17]:
def build_publication_table(df: pd.DataFrame) -> pd.DataFrame:
    """Rename/round combined training+benchmark records into a paper-ready table."""
    if df.empty:
        return df
    rename = {
        "config_id": "Model",
        "acc1": "Top-1 Acc (%)",
        "params_total": "Params",
        "macs": "MACs",
        "bops": "BOPs",
        "latency_ms_b1": "Latency b1 (ms)",
        "peak_memory_mb": "Peak Memory (MB)",
    }
    cols = [c for c in rename if c in df.columns]
    table = df[cols].rename(columns=rename).copy()
    if "Top-1 Acc (%)" in table.columns:
        table = table.sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


publication_df = build_publication_table(combined_df)
publication_df.to_csv(out_dir / "cifar10_published_results.csv", index=False)
publication_df


,Model,Top-1 Acc (%),Params,MACs,BOPs,Latency b1 (ms),Peak Memory (MB)
0,hbcc_accuracy_medium,94.45,2840862,None,0,15.478923,145.565918
1,resnet18_cifar,94.16,11173962,None,0,3.059296,326.364746
2,hbcc_accuracy_small,94.0,1538618,None,0,13.745859,109.592285
3,shufflenet_v2_x1_0_cifar,91.68,1263854,None,0,6.525383,94.832520
4,mobilenet_v2_cifar,91.04,2236682,None,0,5.333046,123.365234
5,coc_cifar_baseline,88.34,2395814,None,0,12.469929,71.798828
6,hbcc_accuracy_small_pruned_export,86.48,1538618,None,0,13.007071,109.592285
7,hbcc_tiny_hamming_late,None,447746,None,5120,7.630177,44.390625
8,hbcc_tiny_lbpconv,None,463258,None,0,7.570573,116.450195
9,hbcc_current_reference,None,870100,None,0,7.676887,173.993164


In [18]:
if RUN_PARETO_REPORT:
    run_py(["tools/pareto_report.py", "results/benchmark", "--output", "results/pareto.md"])


$ /usr/bin/python3 tools/pareto_report.py results/benchmark --output results/pareto.md


results/pareto.md

[exit=0] elapsed=0.7s


## Recommended Execution Order

1. Run environment checks and smoke.
2. Enable `RUN_TEACHER` and train ResNet-18.
3. Enable `RUN_COC_BASELINE` and train the CoC CIFAR baseline.
4. Enable `RUN_STUDENT_CE` and train HBCC Tiny/Small plus HBCC-Accuracy Small/Medium.
5. Enable `RUN_PROXY_SEARCH`; use the table to choose top candidates.
6. Enable `RUN_ABLATIONS` for Hamming/LBPConv.
7. Enable `RUN_KD` after the teacher checkpoint exists.
8. Enable `RUN_PRUNING` after `runs_accuracy/hbcc_accuracy_small/best.pth` exists; this prunes `hbcc_accuracy_small`.
9. Enable `RUN_BENCHMARKS` and build final tables.

Do not treat proxy accuracy or untrained benchmark records as final scientific evidence.